In [2]:
# -*- coding: utf-8 -*-
"""
Формирование датасета YawnDD из видео для пересчета коэффициентов K.

Выполняет:
  - загрузку видео из папки test_videos
  - для каждого видео: извлечение кадров с лицом и расчет MAR
  - классификацию кадров:
      * yawn: MAR >= 0.60 (широко открытый рот — зевок)
      * no_yawn: 0.010 <= MAR <= 0.029 (рот закрыт, нормальная анатомия)
  - сохранение кадров в структуру датасета:
      YawnDD/
          yawn/    (изображения с зевком)
          no_yawn/  (изображения с закрытым ртом)

Автор: Волкова Н.В.
Версия: 1.0 (ВКР)
"""

import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path

# ============================================================
# НАСТРОЙКИ
# ============================================================
VIDEO_DIR = Path("test_videos")
OUTPUT_DATASET_DIR = Path("YawnDD")

YAWN_DIR = OUTPUT_DATASET_DIR / "yawn"
NO_YAWN_DIR = OUTPUT_DATASET_DIR / "no_yawn"

# Пороги классификации
MAR_YAWN_THRESHOLD = 0.60          # MAR >= 0.60 -> зевок
MAR_NO_YAWN_MIN = 0.010            # MAR >= 0.010
MAR_NO_YAWN_MAX = 0.040            # MAR <= 0.040 -> рот закрыт

# Дополнительное условие: глаза открыты
EAR_OPEN_THRESHOLD = 0.20

# Шаг извлечения кадров
FRAME_STEP = 3

# Поддерживаемые расширения
VIDEO_EXTENSIONS = {'.avi', '.mp4', '.mov', '.mkv', '.webm'}

# ============================================================
# СОЗДАНИЕ ПАПОК
# ============================================================
YAWN_DIR.mkdir(parents=True, exist_ok=True)
NO_YAWN_DIR.mkdir(parents=True, exist_ok=True)

print("Настройки загружены.")
print(f"  Папка с видео: {VIDEO_DIR}")
print(f"  Папка для датасета: {OUTPUT_DATASET_DIR}")
print(f"  yawn: MAR >= {MAR_YAWN_THRESHOLD}")
print(f"  no_yawn: {MAR_NO_YAWN_MIN} <= MAR <= {MAR_NO_YAWN_MAX}, EAR > {EAR_OPEN_THRESHOLD}")
print(f"  Шаг кадров: каждый {FRAME_STEP}-й")                                                                               

Настройки загружены.
  Папка с видео: test_videos
  Папка для датасета: YawnDD
  yawn: MAR >= 0.6
  no_yawn: 0.01 <= MAR <= 0.04, EAR > 0.2
  Шаг кадров: каждый 3-й


In [12]:
class VideoFrameExtractor:
    """
    Извлечение кадров из видео и расчет метрик лица.
    """

    MAR_IDX = [13, 14, 78, 308]
    LEFT_EYE_IDX = [362, 385, 387, 263, 373, 380]
    RIGHT_EYE_IDX = [33, 160, 158, 133, 153, 144]

    def __init__(self):
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = None

    def initialize(self):
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    def release(self):
        if self.face_mesh is not None:
            self.face_mesh.close()

    def compute_mar(self, landmarks, w, h):
        top = np.array([landmarks[self.MAR_IDX[0]].x * w,
                        landmarks[self.MAR_IDX[0]].y * h])
        bottom = np.array([landmarks[self.MAR_IDX[1]].x * w,
                           landmarks[self.MAR_IDX[1]].y * h])
        left = np.array([landmarks[self.MAR_IDX[2]].x * w,
                         landmarks[self.MAR_IDX[2]].y * h])
        right = np.array([landmarks[self.MAR_IDX[3]].x * w,
                          landmarks[self.MAR_IDX[3]].y * h])
        vertical = np.linalg.norm(top - bottom)
        horizontal = np.linalg.norm(left - right)
        if horizontal < 1e-6:
            return 0.0
        return max(vertical / horizontal, 0.03)

    def compute_ear(self, landmarks, eye_idx, w, h):
        pts = np.array([[landmarks[i].x * w, landmarks[i].y * h]
                        for i in eye_idx])
        vertical = (np.linalg.norm(pts[1] - pts[5]) +
                    np.linalg.norm(pts[2] - pts[4]))
        horizontal = 2.0 * np.linalg.norm(pts[0] - pts[3])
        return vertical / (horizontal + 1e-6)

In [14]:
class DatasetBuilder:
    """
    Формирование датасета YawnDD из видео.
    """

    def __init__(self, extractor):
        self.extractor = extractor

    def build(self, video_dir, yawn_dir, no_yawn_dir,
              mar_yawn_threshold, mar_no_yawn_min, mar_no_yawn_max,
              ear_open_threshold, frame_step):
        """
        Построение датасета из всех видео в папке.

        Возвращает словарь со статистикой.
        """
        video_paths = []
        for ext in VIDEO_EXTENSIONS:
            video_paths.extend(video_dir.glob(f"*{ext}"))
            video_paths.extend(video_dir.glob(f"*{ext.upper()}"))
        video_paths = sorted(set(video_paths))

        print(f"Найдено видео: {len(video_paths)}")

        total_yawn = 0
        total_no_yawn = 0
        total_faces = 0

        for video_idx, video_path in enumerate(video_paths):
            video_name = video_path.stem
            print(f"\n[{video_idx+1}/{len(video_paths)}] {video_name}")

            cap = cv2.VideoCapture(str(video_path))
            if not cap.isOpened():
                print(f"  Не удалось открыть видео")
                continue

            self.extractor.initialize()

            frame_idx = 0
            saved_yawn = 0
            saved_no_yawn = 0
            faces_detected = 0

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_idx % frame_step != 0:
                    frame_idx += 1
                    continue

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = self.extractor.face_mesh.process(rgb)

                if res.multi_face_landmarks:
                    faces_detected += 1
                    landmarks = res.multi_face_landmarks[0].landmark
                    h, w = frame.shape[:2]

                    mar = self.extractor.compute_mar(landmarks, w, h)
                    ear_l = self.extractor.compute_ear(
                        landmarks, self.extractor.LEFT_EYE_IDX, w, h
                    )
                    ear_r = self.extractor.compute_ear(
                        landmarks, self.extractor.RIGHT_EYE_IDX, w, h
                    )
                    ear = (ear_l + ear_r) / 2.0

                    # Классификация: yawn
                    if mar >= mar_yawn_threshold:
                        filename = (
                            f"{video_name}_f{frame_idx:06d}_mar{mar:.3f}.jpg"
                        )
                        cv2.imwrite(str(yawn_dir / filename), frame)
                        saved_yawn += 1

                    # Классификация: no_yawn
                    elif (mar_no_yawn_min <= mar <= mar_no_yawn_max and
                          ear > ear_open_threshold):
                        filename = (
                            f"{video_name}_f{frame_idx:06d}_mar{mar:.3f}.jpg"
                        )
                        cv2.imwrite(str(no_yawn_dir / filename), frame)
                        saved_no_yawn += 1

                frame_idx += 1

            cap.release()
            self.extractor.release()

            total_yawn += saved_yawn
            total_no_yawn += saved_no_yawn
            total_faces += faces_detected

            print(f"  Кадров с лицом: {faces_detected}")
            print(f"  Сохранено yawn: {saved_yawn}")
            print(f"  Сохранено no_yawn: {saved_no_yawn}")

        return {
            'total_videos': len(video_paths),
            'total_faces': total_faces,
            'total_yawn': total_yawn,
            'total_no_yawn': total_no_yawn
        }

In [15]:
def main():
    """Главная функция."""
    print("=" * 60)
    print("ФОРМИРОВАНИЕ ДАТАСЕТА YawnDD ИЗ ВИДЕО")
    print("=" * 60)

    extractor = VideoFrameExtractor()
    builder = DatasetBuilder(extractor)

    stats = builder.build(
        video_dir=VIDEO_DIR,
        yawn_dir=YAWN_DIR,
        no_yawn_dir=NO_YAWN_DIR,
        mar_yawn_threshold=MAR_YAWN_THRESHOLD,
        mar_no_yawn_min=MAR_NO_YAWN_MIN,
        mar_no_yawn_max=MAR_NO_YAWN_MAX,
        ear_open_threshold=EAR_OPEN_THRESHOLD,
        frame_step=FRAME_STEP
    )

    print("\n" + "=" * 60)
    print("ИТОГИ")
    print("=" * 60)
    print(f"Обработано видео: {stats['total_videos']}")
    print(f"Всего кадров с лицом: {stats['total_faces']}")
    print(f"Сохранено в yawn:    {stats['total_yawn']}")
    print(f"Сохранено в no_yawn: {stats['total_no_yawn']}")
    print(f"\nДатасет сохранен в: {OUTPUT_DATASET_DIR}")
    print(f"  yawn/:    {stats['total_yawn']} изображений")
    print(f"  no_yawn/: {stats['total_no_yawn']} изображений")
    print("=" * 60)


if __name__ == "__main__":
    main()

ФОРМИРОВАНИЕ ДАТАСЕТА YawnDD ИЗ ВИДЕО
Найдено видео: 29

[1/29] 1-FemaleNoGlasses
  Кадров с лицом: 914
  Сохранено yawn: 28
  Сохранено no_yawn: 314

[2/29] 1-MaleGlasses
  Кадров с лицом: 859
  Сохранено yawn: 99
  Сохранено no_yawn: 209

[3/29] 10-FemaleNoGlasses
  Кадров с лицом: 404
  Сохранено yawn: 24
  Сохранено no_yawn: 30

[4/29] 10-MaleGlasses
  Кадров с лицом: 806
  Сохранено yawn: 56
  Сохранено no_yawn: 151

[5/29] 11-FemaleGlasses.avi
  Кадров с лицом: 561
  Сохранено yawn: 74
  Сохранено no_yawn: 143

[6/29] 11-MaleGlasses
  Кадров с лицом: 587
  Сохранено yawn: 55
  Сохранено no_yawn: 75

[7/29] 12-FemaleGlasses.avi
  Кадров с лицом: 838
  Сохранено yawn: 25
  Сохранено no_yawn: 279

[8/29] 12-MaleGlasses
  Кадров с лицом: 653
  Сохранено yawn: 60
  Сохранено no_yawn: 54

[9/29] 13-FemaleGlasses.avi
  Кадров с лицом: 830
  Сохранено yawn: 19
  Сохранено no_yawn: 106

[10/29] 13-MaleNoGlasses 
  Кадров с лицом: 730
  Сохранено yawn: 44
  Сохранено no_yawn: 482

[11/29] 